In [17]:
from lmut import LMUT
from dqn import DQNAgent, DQNNetwork, play_episode
from visualization import show_animation
from IPython.display import HTML
import gymnasium as gym
import numpy as np
import os


In [2]:
models_dir = "models"
dqn_path = os.path.join(models_dir, "dqn_cartpole")
lmut_path = os.path.join(models_dir, "lmut")


In [3]:
env = gym.make("CartPole-v1", render_mode="rgb_array")
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

print(state_dim, n_actions)


4 2


In [4]:
agent = DQNAgent(state_dim, n_actions)
agent.load(dqn_path)


In [5]:
episode_reward, episode_steps, _ = play_episode(env, agent)
print(f"Recompensa del episodio: {episode_reward}")
print(f"Pasos del episodio: {episode_steps}")


c:\Users\malos\Documents\GitHub\XRL\.venv\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Recompensa del episodio: 500.0
Pasos del episodio: 500


In [6]:
def collect_qs(env: gym.Env, agent: DQNAgent, episodes: int = 100, epsilon: float = 0.1):
	qs = []    

	for _ in range(episodes):
		state, _ = env.reset()
		done = False

		while not done:
			action = agent.epsilon_greedy(state, epsilon)
			q_values = agent.predict(state)

			qs.extend(q_values)

			state, _, terminated, truncated, _ = env.step(action)
			done = terminated or truncated

	return np.array(qs)


In [7]:
qs = collect_qs(env, agent)
print(f"min={qs.min():.2f}  max={qs.max():.4f}  mean={qs.mean():.4f}  std={qs.std():.4f}")
print(f"percentiles 1/50/99: {np.percentile(qs, [1, 50, 99])}")


KeyboardInterrupt: 

In [8]:
def active_play_mimic(env, agent: DQNAgent, mimic_lmut: LMUT, episodes: int = 200, train_every: int = 200, epsilon_start: float = 1.0, epsilon_end: float = 0.01, decay: float = 0.995):
	epsilon = epsilon_start
	buffer = []

	for ep in range(episodes):
		state, _ = env.reset()
		done = False
		while not done:
			action = agent.epsilon_greedy(state, epsilon)
			teacher_q = agent.predict(state)[action]

			buffer.append((state, action, teacher_q))

			next_state, _, terminated, truncated, _ = env.step(action)
			done = terminated or truncated
			state = next_state

			if len(buffer) >= train_every:
				for s, a, tq in buffer:
					mimic_lmut.add_transition(s, a, tq)
				stats = mimic_lmut.update_all_leaves()
				mse = mimic_lmut.compute_current_mse()

				print(
					f"Episode {ep+1} | "
					f"Leaves: {stats['leaf_count']} | "
					# f"AvgLoss: {stats['avg_loss']:.4f} | "
					f"MSE: {mse:.4f} | "
					f"Splits: {stats['splits']} | "
					f"ε: {epsilon:.3f}"
				)
				
				buffer.clear()

		epsilon = max(epsilon_end, epsilon * decay)
		

In [5]:
mimic = LMUT(state_dim, n_actions, q_mean=qs.mean(), q_std=qs.std(), lr=0.01)


NameError: name 'qs' is not defined

In [12]:
active_play_mimic(env, agent, mimic, episodes=300)


Episode 8 | Leaves: 2 | MSE: 6841.6045 | Splits: 0 | ε: 0.966
Episode 14 | Leaves: 2 | MSE: 5170.9180 | Splits: 2 | ε: 0.937
Episode 21 | Leaves: 4 | MSE: 6092.4038 | Splits: 2 | ε: 0.905
Episode 31 | Leaves: 6 | MSE: 7089.7983 | Splits: 1 | ε: 0.860
Episode 40 | Leaves: 7 | MSE: 7333.9922 | Splits: 2 | ε: 0.822
Episode 47 | Leaves: 9 | MSE: 6849.3555 | Splits: 1 | ε: 0.794
Episode 50 | Leaves: 10 | MSE: 6096.8086 | Splits: 1 | ε: 0.782
Episode 54 | Leaves: 11 | MSE: 5479.3657 | Splits: 1 | ε: 0.767
Episode 59 | Leaves: 12 | MSE: 5242.1484 | Splits: 1 | ε: 0.748
Episode 63 | Leaves: 13 | MSE: 4953.5352 | Splits: 1 | ε: 0.733
Episode 70 | Leaves: 14 | MSE: 4804.7866 | Splits: 1 | ε: 0.708
Episode 75 | Leaves: 15 | MSE: 4541.6216 | Splits: 0 | ε: 0.690
Episode 80 | Leaves: 15 | MSE: 4275.6064 | Splits: 1 | ε: 0.673
Episode 84 | Leaves: 16 | MSE: 4004.5256 | Splits: 1 | ε: 0.660
Episode 90 | Leaves: 17 | MSE: 3605.7856 | Splits: 0 | ε: 0.640
Episode 97 | Leaves: 17 | MSE: 3430.2131 | Spli

In [13]:
mimic.print_tree(0)


=== Tree for action 0 ===
[Node] depth=0 | split: feature 2 < 0.0339
  left:
    [Node] depth=1 | split: feature 3 < -0.5969
      left:
        [Node] depth=2 | split: feature 2 < -0.1404
          left:
            [Node] depth=3 | split: feature 3 < -1.1224
              left:
                [Leaf] depth=4 | y = [-0.25606263 -2.9763227   0.8535153   5.4411955 ] * s + -3.4513
              right:
                [Node] depth=4 | split: feature 2 < -0.1747
                  left:
                    [Leaf] depth=5 | y = [-0.28694424 -1.7148706   0.9133243   5.064696  ] * s + -2.5158
                  right:
                    [Leaf] depth=5 | y = [-0.7009262 -1.2969786  0.7360002  4.297541 ] * s + -1.2241
          right:
            [Node] depth=3 | split: feature 3 < -1.1691
              left:
                [Node] depth=4 | split: feature 2 < -0.0955
                  left:
                    [Leaf] depth=5 | y = [ 0.0769522  -0.7658461   0.36850435  1.3220507 ] * s + -0.4749

In [14]:
mimic.save(lmut_path)


In [8]:
lmut2 = LMUT.load(lmut_path)


In [9]:
lmut2.print_tree(0)



=== Tree for action 0 ===
[Node] depth=0 | split: feature 2 < 0.0339
  left:
    [Node] depth=1 | split: feature 3 < -0.5969
      left:
        [Node] depth=2 | split: feature 2 < -0.1404
          left:
            [Node] depth=3 | split: feature 3 < -1.1224
              left:
                [Leaf] depth=4 | y = [-0.25606263 -2.9763227   0.8535153   5.4411955 ] * s + -3.4513
              right:
                [Node] depth=4 | split: feature 2 < -0.1747
                  left:
                    [Leaf] depth=5 | y = [-0.28694424 -1.7148706   0.9133243   5.064696  ] * s + -2.5158
                  right:
                    [Leaf] depth=5 | y = [-0.7009262 -1.2969786  0.7360002  4.297541 ] * s + -1.2241
          right:
            [Node] depth=3 | split: feature 3 < -1.1691
              left:
                [Node] depth=4 | split: feature 2 < -0.0955
                  left:
                    [Leaf] depth=5 | y = [ 0.0769522  -0.7658461   0.36850435  1.3220507 ] * s + -0.4749

In [10]:
def evaluate_fidelity(env: gym.Env, teacher: DQNAgent, mimic_lmut: LMUT, num_samples: int = 1000):
	mae_sum = 0.0
	mse_sum = 0.0
	count = 0
	correct = 0

	state, _ = env.reset()
	for _ in range(num_samples):
		q_all = teacher.predict(state)
		teacher_action = int(np.argmax(q_all))
		teacher_q = q_all[teacher_action]
	
		mimic_qs = np.array(mimic_lmut.predict_all_actions(state))
		
		mimic_action = np.argmax(mimic_qs)
		mimic_q = mimic_qs[teacher_action]

		if teacher_action == mimic_action:
			correct += 1

		error = teacher_q - mimic_q
		mae_sum += abs(error)
		mse_sum += error ** 2
		count += 1

		next_state, _, terminated, truncated, _ = env.step(teacher_action)
		if terminated or truncated:
			state, _ = env.reset()
		else:
			state = next_state

	mae = mae_sum / count
	rmse = (mse_sum / count) ** 0.5
	accuracy = correct / count
	return mae, rmse, accuracy


In [11]:
mae, rmse, acc = evaluate_fidelity(env, agent, lmut2, num_samples=1000)
print(f"Fidelity: MAE = {mae:.4f}, RMSE = {rmse:.4f}, Action Accuracy = {acc:.2%}")


Fidelity: MAE = 1.6375, RMSE = 1.9822, Action Accuracy = 52.00%


In [12]:
def play_episode_lmut(env: gym.Env, lmut: LMUT):
    frames = []
    state, _ = env.reset()

    episode_steps = episode_reward = 0
    done = False
    while not done:
        frames.append(env.render())
        q_values = np.array(lmut.predict_all_actions(state))
        action = int(np.argmax(q_values))

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        episode_steps += 1
        episode_reward += reward

        state = next_state
    return episode_reward, episode_steps, frames



In [ ]:
print("\n=== Feature Influence ===")
feature_names = ["Cart Position", "Cart Velocity", "Pole Angle", "Pole Angular Velocity"]
for i, inf in enumerate(lmut2.feature_influence):
	print(f"{feature_names[i]} : {inf:.6f}")
	


=== Feature Influence ===
Cart Position        : 11.449128
Cart Velocity        : 6.113397
Pole Angle           : 243.934555
Pole Angular Velocity : 636.434060


In [15]:
episode_reward, episode_steps, frames = play_episode_lmut(env, lmut2)
print(f"Recompensa del episodio: {episode_reward}")
print(f"Pasos del episodio: {episode_steps}")


c:\Users\malos\Documents\GitHub\XRL\.venv\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Recompensa del episodio: 208.0
Pasos del episodio: 208


In [18]:
ani = show_animation(frames)
HTML(ani.to_jshtml())
